In [3]:
import os
import json
import ast  # Para interpretar o conteúdo do 'tool_call'

def compute_averages_per_file(paths):
    common_files = set(os.listdir(paths[0]))
    for path in paths[1:]:
        common_files &= set(os.listdir(path))

    results = {}

    for file_name in sorted(common_files):
        total_qc_words = 0
        total_qc_chars = 0
        total_resp_words = 0
        total_resp_chars = 0
        total_entries = 0

        total_tool_calls = 0
        total_tool_errors = 0

        for path in paths:
            with open(os.path.join(path, file_name), 'r', encoding='utf-8') as f:
                data = json.load(f)
                for item in data.get('results', []):
                    # Processa question + context e response
                    question_context = item['question'] + ' ' + item['context']
                    response = item['response']

                    total_qc_words += len(question_context.split())
                    total_qc_chars += len(question_context)
                    total_resp_words += len(response.split())
                    total_resp_chars += len(response)
                    total_entries += 1

                    # Processa tool_call
                    tool_call_str = item.get('tool_call', '')
                    if tool_call_str:
                        # Conta quantos ToolCallResult existem (número de chamadas)
                        num_calls = tool_call_str.count('ToolCallResult')
                        total_tool_calls += num_calls

                        # Conta quantos is_error=True existem (erros)
                        num_errors = tool_call_str.count('is_error=True')
                        total_tool_errors += num_errors

        # Calcula médias
        avg_qc_words = total_qc_words / total_entries if total_entries else 0
        avg_qc_chars = total_qc_chars / total_entries if total_entries else 0
        avg_resp_words = total_resp_words / total_entries if total_entries else 0
        avg_resp_chars = total_resp_chars / total_entries if total_entries else 0

        avg_tool_calls = total_tool_calls / total_entries if total_entries else 0
        tool_call_accuracy = ((total_tool_calls - total_tool_errors) / total_tool_calls * 100) if total_tool_calls else 0

        results[file_name] = {
            'avg_qc_words': avg_qc_words,
            'avg_qc_chars': avg_qc_chars,
            'avg_resp_words': avg_resp_words,
            'avg_resp_chars': avg_resp_chars,
            'avg_tool_calls': avg_tool_calls,
            'tool_calls_total': total_tool_calls,
            'tool_calls_errors': total_tool_errors,
            'tool_calls_accuracy_percent': tool_call_accuracy
        }

    # Exibe resultados
    for file_name, stats in results.items():
        print(f"\nFile: {file_name}")
        print(f"  Average words in question + context: {stats['avg_qc_words']:.2f}")
        print(f"  Average characters in question + context: {stats['avg_qc_chars']:.2f}")
        print(f"  Average words in response: {stats['avg_resp_words']:.2f}")
        print(f"  Average characters in response: {stats['avg_resp_chars']:.2f}")
        print(f"  Average function calls per entry: {stats['avg_tool_calls']:.2f}")
        print(f"  Total function calls: {stats['tool_calls_total']}")
        print(f"  Total function call errors: {stats['tool_calls_errors']}")
        print(f"  Function call accuracy: {stats['tool_calls_accuracy_percent']:.2f}%")



In [4]:
# Exemplo de uso
paths = ['run1_corrected',
         'run2_corrected', 
         'run3_corrected']
compute_averages_per_file(paths)


File: command-r7b_output.json
  Average words in question + context: 4819.00
  Average characters in question + context: 30433.84
  Average words in response: 129.80
  Average characters in response: 831.13
  Average function calls per entry: 0.17
  Total function calls: 19
  Total function call errors: 15
  Function call accuracy: 21.05%

File: deepseek-r1_14b_output.json
  Average words in question + context: 4819.00
  Average characters in question + context: 30433.84
  Average words in response: 71.70
  Average characters in response: 455.25
  Average function calls per entry: 0.19
  Total function calls: 22
  Total function call errors: 0
  Function call accuracy: 100.00%

File: deepseek-r1_32b_output.json
  Average words in question + context: 4819.00
  Average characters in question + context: 30433.84
  Average words in response: 85.66
  Average characters in response: 541.02
  Average function calls per entry: 0.05
  Total function calls: 6
  Total function call errors: 0
  F